In [1]:
import pandas as pd

In [2]:
df = pd.read_parquet("../data/candidate_detail.parquet")

In [3]:
pd.set_option("display.max_rows", None)

In [4]:
c = pd.read_parquet("../data/candidate_detail.parquet")
c = c[c.outcome.isin(["Approved","Commercialized","Failed Phase 1","Failed Phase 2","Failed Phase 3"])].copy()
c["y"]     = c.outcome.isin(["Approved","Commercialized"]).astype(int)
c["drug"]  = c.candidate_id.str.split("__").str[0]          # drug identity (memorization key)
c["year"]  = pd.to_datetime(c.earliest_start_date, errors="coerce").dt.year
c["split"] = (c.year <= 2019).map({True:"train", False:"test"})

In [5]:
# join the model's test predictions to inspect seen/new-drug behavior:
pred = pd.read_parquet("../runs/abl_md/predictions.parquet")  # candidate_id, y_true, y_proba
df = pred.merge(c[["candidate_id","drug","outcome","year"]], right_on="candidate_id", left_on="example_id", how="left")
df["seen_drug"] = df.drug.isin(set(c[c.split=="train"].drug))

In [6]:
df.seen_drug.value_counts()

seen_drug
True     2378
False     259
Name: count, dtype: int64

In [ ]:
groups = df[df.year > 2019].groupby("drug")
for drug, group in groups:
    if len(group) == 1: 
        continue
    #if group.label.sum() == len(group) or group.label.sum() == 0:
        #continue
    if group.label.sum() != 0:
        continue
    # if group.seen_drug.all():
    #     continue

    weakest_approval = group[group.label==1].y_proba.min()
    strongest_failure = group[group.label==0].y_proba.max()
    if weakest_approval < strongest_failure:
        continue
    print(f"Drug: {drug}")
    print(group[["example_id","outcome","label","y_proba","seen_drug"]])
    print("-" * 40)


Drug: db:DB00162
                     example_id         outcome  label   y_proba  seen_drug
103         db:DB00162__anosmia  Failed Phase 2      0  0.255197       True
2400  db:DB00162__acne vulgaris        Approved      1  0.255197       True
----------------------------------------
Drug: db:DB00176
                              example_id         outcome  label   y_proba  \
766                 db:DB00176__delirium        Approved      1  0.461845   
1141  db:DB00176__coronavirus infections  Failed Phase 3      0  0.341274   

      seen_drug  
766        True  
1141       True  
----------------------------------------
Drug: db:DB00177
                                          example_id         outcome  label  \
794   db:DB00177__st elevation myocardial infarction        Approved      1   
870                  db:DB00177__acute kidney injury  Failed Phase 3      0   
1214                            db:DB00177__covid-19  Failed Phase 3      0   
1368              db:DB00177__resista

In [11]:
df[df.example_id.str.contains("exenatide")]

,example_id,label,phase,y_proba,candidate_id,drug,outcome,year,seen_drug


In [7]:
# ROC AUC on seen vs. new drugs:
from sklearn.metrics import roc_auc_score
print("ROC AUC on seen drugs:", roc_auc_score(df[df.seen_drug].label, df[df.seen_drug].y_proba))
print("ROC AUC on new drugs:", roc_auc_score(df[~df.seen_drug].label, df[~df.seen_drug].y_proba))

ROC AUC on seen drugs: 0.5948137223209956
ROC AUC on new drugs: 0.5760854953622798


In [8]:
# PR AUC on seen vs. new drugs:
from sklearn.metrics import average_precision_score
print("PR AUC on seen drugs:", average_precision_score(df[df.seen_drug].label, df[df.seen_drug].y_proba))
print("PR AUC on new drugs:", average_precision_score(df[~df.seen_drug].label, df[~df.seen_drug].y_proba))

PR AUC on seen drugs: 0.3212923759901194
PR AUC on new drugs: 0.3962363113148435
